# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [2]:
import pandas as pd
import requests
import io
import urllib3

# Olası kırmızı güvenlik uyarılarını gizleyelim
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Veriyi internetten güvenlik kontrolünü atlayarak indirelim
url = "https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv"
response = requests.get(url, verify=False)

# İndirdiğimiz ham veriyi Pandas'a tablo olarak okutalım
df = pd.read_csv(io.StringIO(response.text))

# İşte tablomuz!
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [3]:
# Her kalite puanından (quality rating) kaç gözlem olduğunu görelim
df['quality rating'].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [4]:
# Hedef değişkenimiz y'yi oluşturalım: 6 ve üstü -> 1, 6 altı -> 0
y = (df['quality rating'] >= 6).astype(int)

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [5]:
# Yeni ikili hedefin sınıf dağılımına bakalım
y.value_counts()

quality rating
0    50010
1    49990
Name: count, dtype: int64

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [6]:
# Sadece özellikleri (features) içeren X_raw veri çerçevesini oluşturalım
X_raw = df.drop(columns=['quality rating'])

In [7]:
from sklearn.preprocessing import StandardScaler

# StandardScaler ile veriyi aynı ölçeğe (ortalama=0, standart sapma=1) getirelim
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
import pandas as pd

solvers = ['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
yaris_sonuclari = []

for s in solvers:
    # Modeli ilgili çözücüyle kuralım (max_iter=1000 veriyoruz ki uyarı vermesin)
    model = LogisticRegression(solver=s, max_iter=1000, random_state=42)
    
    # Çapraz doğrulama ile hem süreyi (fit_time) hem kesinliği (test_score) ölçelim
    cv_results = cross_validate(model, X, y, cv=5, scoring='accuracy')
    
    # Sonuçları kaydedelim
    yaris_sonuclari.append({
        'Çözücü (Solver)': s,
        'Süre (saniye)': cv_results['fit_time'].mean(),
        'Kesinlik (Accuracy)': cv_results['test_score'].mean()
    })

# Sonuçları tabloya çevirip en hızlıdan yavaşa doğru sıralayalım
df_sonuclar = pd.DataFrame(yaris_sonuclari).sort_values(by='Süre (saniye)')
display(df_sonuclar)

,Çözücü (Solver),Süre (saniye),Kesinlik (Accuracy)
1,lbfgs,0.078282,0.86108
0,newton-cg,0.153382,0.86110
2,liblinear,0.207583,0.86108
3,sag,2.208491,0.86111
4,saga,2.928131,0.86111


In [9]:
# YOUR ANSWER
fastest_solver = "liblinear"

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [10]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D4-S-data-solvers/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.02s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [11]:
from sklearn.linear_model import SGDClassifier

# loss='log_loss' demek, SGD'ye Lojistik Regresyon gibi davranmasını söyler
sgd_model = SGDClassifier(loss='log_loss', random_state=42)

# Süreyi ve kesinliği ölçelim
cv_results_sgd = cross_validate(sgd_model, X, y, cv=5, scoring='accuracy')

print(f"SGD Ortalama Süresi: {cv_results_sgd['fit_time'].mean():.4f} saniye")
print(f"SGD Kesinlik (Accuracy): {cv_results_sgd['test_score'].mean():.4f}")

SGD Ortalama Süresi: 0.2084 saniye
SGD Kesinlik (Accuracy): 0.8601


☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [13]:
import pandas as pd
import requests
import io
import urllib3

# Mac SSL hatasını aşarak yeni şarabın verisini indirelim
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
url_new_wine = 'https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv'
response_new = requests.get(url_new_wine, verify=False)
new_wine = pd.read_csv(io.StringIO(response_new.text))

# Yeni şarabın özelliklerini görelim
new_wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol
0,9.54,13.5,12.35,8.78,14.72,9.06,9.67,10.15,11.17,12.17


In [14]:
# 1. En iyi modelimiz olan 'liblinear'ı tüm veriyle eğitelim
best_model = LogisticRegression(solver='liblinear', random_state=42)
best_model.fit(X, y)

# 2. Yeni şarabın özelliklerini standart ölçeğe getirelim (scaler'ı yukarıda tanımlamıştık)
X_new = scaler.transform(new_wine)

# 3. Tahmin yapalım ve hocanın istediği değişkenlere kaydedelim
predicted_class = best_model.predict(X_new)[0]

# Modelin tahminine ne kadar güvendiğini (olasılığını) bulalım
proba_array = best_model.predict_proba(X_new)[0]
predicted_proba_of_class = proba_array[predicted_class]

print(f"Tahmin Edilen Sınıf: {predicted_class} (1: İyi, 0: Kötü)")
print(f"Emin Olma Olasılığı: {predicted_proba_of_class:.4f}")

Tahmin Edilen Sınıf: 0 (1: İyi, 0: Kötü)
Emin Olma Olasılığı: 0.9687


# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [15]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/macos/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /Users/macos/Desktop/S16D4-S-data-solvers/tests
plugins: dash-4.0.0, anyio-4.8.0, typeguard-4.4.2
collecting ... collected 2 items

test_new_data_prediction.py::TestNewDataPrediction::test_predicted_class PASSED [ 50%]
test_new_data_prediction.py::TestNewDataPrediction::test_predicted_proba PASSED [100%]

============================== 2 passed in 0.23s ===============================


💯 You can commit your code:

git add tests/new_data_prediction.pickle

git commit -m 'Completed new_data_prediction step'

git push origin master

